# Milestone 3 – CNN on SVHN (No Code Smells)

Baseline CNN architecture from Milestone 2 trained on the **SVHN** dataset.
- EarlyStopping patience=2
- 10 independent runs
- CodeCarbon energy & CO₂ tracking

In [ ]:
!pip install -q codecarbon scipy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc, os, warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dense, Dropout, BatchNormalization, Flatten
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K

from sklearn.metrics import classification_report, confusion_matrix
from codecarbon import EmissionsTracker


In [ ]:
import tensorflow_datasets as tfds

print("Downloading SVHN dataset via tensorflow_datasets...")
(ds_train, ds_test), ds_info = tfds.load(
    'svhn_cropped',
    split=['train', 'test'],
    shuffle_files=False,
    as_supervised=True,
    with_info=True
)

def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

ds_train = ds_train.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
ds_test  = ds_test.map(preprocess,  num_parallel_calls=tf.data.AUTOTUNE)

# Convert to numpy arrays for model.fit compatibility
X_train = np.stack([img.numpy() for img, _ in ds_train])
y_train = np.array([lbl.numpy() for _, lbl in ds_train])
X_test  = np.stack([img.numpy() for img, _ in ds_test])
y_test  = np.array([lbl.numpy() for _, lbl in ds_test])

y_cat_train = to_categorical(y_train, 10)
y_cat_test  = to_categorical(y_test,  10)

print(f"X_train: {X_train.shape}  X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}  y_test: {y_test.shape}")


In [ ]:
INPUT_SHAPE = (32, 32, 3)
KERNEL_SIZE = (3, 3)

def build_cnn():
    METRICS = [
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
    model = Sequential()

    # Block 1 – 32 filters
    model.add(Conv2D(filters=32, kernel_size=KERNEL_SIZE, input_shape=INPUT_SHAPE, activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(Conv2D(filters=32, kernel_size=KERNEL_SIZE, activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(MaxPool2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    # Block 2 – 64 filters
    model.add(Conv2D(filters=64, kernel_size=KERNEL_SIZE, activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(Conv2D(filters=64, kernel_size=KERNEL_SIZE, activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(MaxPool2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    # Block 3 – 128 filters
    model.add(Conv2D(filters=128, kernel_size=KERNEL_SIZE, activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(Conv2D(filters=128, kernel_size=KERNEL_SIZE, activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(MaxPool2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    # Fully connected
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.25))
    model.add(Dense(10, activation='softmax'))

    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=METRICS)
    return model


## Training – 10 Runs

In [ ]:
os.makedirs("emissions", exist_ok=True)
results = []

for run in range(1, 11):
    print(f"\n{'='*60}")
    print(f"  Run {run}/10")
    print(f"{'='*60}")

    # Clean practice: reset graph between runs
    K.clear_session()

    tracker = EmissionsTracker(
        project_name=f"CNN_SVHN_Clean_Run{run}",
        output_dir="./emissions",
        save_to_file=True,
        log_level="warning"
    )
    tracker.start()

    model      = build_cnn()
    early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

    history = model.fit(
        X_train, y_cat_train,
        epochs=50, batch_size=32,
        validation_data=(X_test, y_cat_test),
        callbacks=[early_stop],
        verbose=1
    )

    co2_kg     = tracker.stop()
    energy_kwh = tracker.final_emissions_data.energy_consumed

    epoch_stopped = len(history.history['loss'])
    eval_res      = model.evaluate(X_test, y_cat_test, verbose=0)
    # eval_res: [loss, accuracy, precision, recall]

    run_result = dict(
        run=run,
        epoch_stopped=epoch_stopped,
        test_loss=eval_res[0],
        test_accuracy=eval_res[1],
        test_precision=eval_res[2],
        test_recall=eval_res[3],
        energy_kwh=energy_kwh,
        co2_kg=co2_kg
    )
    results.append(run_result)
    print(f"Run {run} finished | Epoch stopped: {epoch_stopped} | "
          f"Acc: {eval_res[1]:.4f} | Energy: {energy_kwh:.6f} kWh | CO2: {co2_kg:.6f} kg")

    # Clean practice: free memory explicitly
    del model
    gc.collect()


## Results Summary

In [ ]:
print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
header = f"{'Run':<5} {'EpochStopped':<14} {'Accuracy':<12} {'Precision':<12} {'Recall':<10} {'Energy(kWh)':<14} {'CO2(kg)':<12}"
print(header)
print("-"*80)
for r in results:
    print(f"{r['run']:<5} {r['epoch_stopped']:<14} {r['test_accuracy']:<12.4f} "
          f"{r['test_precision']:<12.4f} {r['test_recall']:<10.4f} "
          f"{r['energy_kwh']:<14.6f} {r['co2_kg']:<12.6f}")
print("-"*80)
avg_epoch  = np.mean([r['epoch_stopped']    for r in results])
avg_acc    = np.mean([r['test_accuracy']    for r in results])
avg_prec   = np.mean([r['test_precision']   for r in results])
avg_rec    = np.mean([r['test_recall']      for r in results])
avg_energy = np.mean([r['energy_kwh']       for r in results])
avg_co2    = np.mean([r['co2_kg']           for r in results])
print(f"{'AVG':<5} {avg_epoch:<14.1f} {avg_acc:<12.4f} {avg_prec:<12.4f} "
      f"{avg_rec:<10.4f} {avg_energy:<14.6f} {avg_co2:<12.6f}")
print("="*80)
